### Set up

In [1]:
from pathlib import Path
import os
import sys

# Find project root by walking upward until we find the src folder
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Set working directory to project root
os.chdir(PROJECT_ROOT)

# Add src folder to Python path
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_PATH)

Project root: /Users/Local/REPOSITORIES/injury_risk_classifier
Source path: /Users/Local/REPOSITORIES/injury_risk_classifier/src


In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

from src import features
from src import clean
from src import audit
from src import geo

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "interim" / "traffic_clean.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(282287, 18)

# Features

## Vehicle type difference

In [4]:
df["vehicle_type_match"] = df.apply(features.vehicle_match, axis=1)

In [5]:
df = features.vehicle_sizes(df)

In [6]:
features.size_relation(df)

,top_traffic_accident_offense,first_occurrence_date,incident_address,lon,lat,road_description,road_condition,light_condition,datetime,date,...,tu1_driver_action_binned,tu2_driver_action_binned,tu1_human_factor_binned,tu2_human_factor_binned,vehicle_type_match,tu1_vehicle_size,tu2_vehicle_size,vehicle_size_diff,abs_vehicle_size_diff,vehicle_size_relation
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,non-intersection,dry,daylight,2022-12-09 02:05:59.999991956,2022-12-09,...,no_action,no_action,no_apparent,no_apparent,different,3.0,4.0,-1.0,1.0,vehicle_2_larger
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,at intersection,dry,daylight,2022-12-11 23:39:59.999991056,2022-12-11,...,failure_to_yield,no_action,no_apparent,no_apparent,different,2.0,3.0,-1.0,1.0,vehicle_2_larger
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,non-intersection,dry,daylight,2022-12-12 00:10:00.000004471,2022-12-12,...,aggressive_or_careless,no_action,no_apparent,no_apparent,same,2.0,2.0,0.0,0.0,same_size
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,non-intersection,dry,dark-lighted,2022-12-09 06:26:59.999983903,2022-12-09,...,aggressive_or_careless,no_action,aggressive_or_evading,no_apparent,different,3.0,2.0,1.0,1.0,vehicle_1_larger
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,non-intersection,dry,dark-lighted,2022-12-09 07:30:00.000000000,2022-12-09,...,unknown,unknown,unknown,unknown,unknown,NaN,2.0,NaN,NaN,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282282,TRAF - ACCIDENT,2.459920e+06,I25 HWYSB / W 23RD AVE,-105.016793,39.751285,at intersection,dry,daylight,2022-12-06 20:00:00.000013415,2022-12-06,...,lane_or_position,no_action,distracted,no_apparent,same,2.0,2.0,0.0,0.0,same_size
282283,TRAF - ACCIDENT - HIT & RUN,2.459942e+06,I70 HWYWB / N MONACO ST,-104.912882,39.778408,non-intersection,dry,daylight,2022-12-29 02:15:00.000000000,2022-12-29,...,aggressive_or_careless,no_action,aggressive_or_evading,no_apparent,different,3.0,3.0,0.0,0.0,same_size
282284,TRAF - ACCIDENT,2.459942e+06,50 BLOCK N HOLLY ST,-104.922176,39.717045,non-intersection,wet,daylight,2022-12-29 02:29:59.999986584,2022-12-29,...,aggressive_or_careless,no_action,no_apparent,no_apparent,different,2.0,3.0,-1.0,1.0,vehicle_2_larger
282285,TRAF - ACCIDENT - HIT & RUN,2.459945e+06,20TH ST / ARAPAHOE ST,-104.990832,39.751590,at intersection,dry,dark-lighted,2022-12-31 14:00:00.000013415,2022-12-31,...,lane_or_position,no_action,no_apparent,no_apparent,same,2.0,2.0,0.0,0.0,same_size


## Actions by vehicle size

In [7]:
features.vehicle_size_actions(df)

,top_traffic_accident_offense,first_occurrence_date,incident_address,lon,lat,road_description,road_condition,light_condition,datetime,date,...,tu1_human_factor_binned,tu2_human_factor_binned,vehicle_type_match,tu1_vehicle_size,tu2_vehicle_size,vehicle_size_diff,abs_vehicle_size_diff,vehicle_size_relation,smaller_vehicle_action,larger_vehicle_action
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,non-intersection,dry,daylight,2022-12-09 02:05:59.999991956,2022-12-09,...,no_apparent,no_apparent,different,3.0,4.0,-1.0,1.0,vehicle_2_larger,no_action,no_action
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,at intersection,dry,daylight,2022-12-11 23:39:59.999991056,2022-12-11,...,no_apparent,no_apparent,different,2.0,3.0,-1.0,1.0,vehicle_2_larger,failure_to_yield,no_action
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,non-intersection,dry,daylight,2022-12-12 00:10:00.000004471,2022-12-12,...,no_apparent,no_apparent,same,2.0,2.0,0.0,0.0,same_size,same_size,same_size
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,non-intersection,dry,dark-lighted,2022-12-09 06:26:59.999983903,2022-12-09,...,aggressive_or_evading,no_apparent,different,3.0,2.0,1.0,1.0,vehicle_1_larger,no_action,aggressive_or_careless
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,non-intersection,dry,dark-lighted,2022-12-09 07:30:00.000000000,2022-12-09,...,unknown,unknown,unknown,NaN,2.0,NaN,NaN,unknown,unknown,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282282,TRAF - ACCIDENT,2.459920e+06,I25 HWYSB / W 23RD AVE,-105.016793,39.751285,at intersection,dry,daylight,2022-12-06 20:00:00.000013415,2022-12-06,...,distracted,no_apparent,same,2.0,2.0,0.0,0.0,same_size,same_size,same_size
282283,TRAF - ACCIDENT - HIT & RUN,2.459942e+06,I70 HWYWB / N MONACO ST,-104.912882,39.778408,non-intersection,dry,daylight,2022-12-29 02:15:00.000000000,2022-12-29,...,aggressive_or_evading,no_apparent,different,3.0,3.0,0.0,0.0,same_size,same_size,same_size
282284,TRAF - ACCIDENT,2.459942e+06,50 BLOCK N HOLLY ST,-104.922176,39.717045,non-intersection,wet,daylight,2022-12-29 02:29:59.999986584,2022-12-29,...,no_apparent,no_apparent,different,2.0,3.0,-1.0,1.0,vehicle_2_larger,aggressive_or_careless,no_action
282285,TRAF - ACCIDENT - HIT & RUN,2.459945e+06,20TH ST / ARAPAHOE ST,-104.990832,39.751590,at intersection,dry,dark-lighted,2022-12-31 14:00:00.000013415,2022-12-31,...,no_apparent,no_apparent,same,2.0,2.0,0.0,0.0,same_size,same_size,same_size


## sidenote - date types 

In [8]:
df = clean.split_datetime(df, "first_occurrence_date")

df = features.create_time_variables(df)

In [9]:
df[[
    "first_occurrence_date",
    "datetime",
    "date",
    "time",
    "hour",
    "is_night",
    "evening_rush",
    "morning_rush"
]].sample(20)

,first_occurrence_date,datetime,date,time,hour,is_night,evening_rush,morning_rush
68759,2.457954e+06,2017-07-19 13:50:00.000008943,2017-07-19,13:50:00,13,False,False,False
12817,2.460050e+06,2023-04-15 00:00:00.000000000,2023-04-15,00:00:00,0,True,False,False
231403,2.459819e+06,2022-08-27 20:17:59.999989274,2022-08-27,20:17:59,20,True,False,False
215419,2.459502e+06,2021-10-14 12:53:00.000011624,2021-10-14,12:53:00,12,False,False,False
76763,2.457763e+06,2017-01-09 12:45:59.999996427,2017-01-09,12:45:59,12,False,False,False
95458,2.458457e+06,2018-12-04 16:30:00.000000000,2018-12-04,16:30:00,16,False,True,False
196622,2.458583e+06,2019-04-09 06:45:00.000000000,2019-04-09,06:45:00,6,False,False,False
194121,2.458641e+06,2019-06-06 15:11:00.000000890,2019-06-06,15:11:00,15,False,False,False
194941,2.458669e+06,2019-07-04 20:00:00.000013415,2019-07-04,20:00:00,20,True,False,False
70159,2.457878e+06,2017-05-04 21:47:59.999989274,2017-05-04,21:47:59,21,True,False,False


In [10]:
timecheck = ((df["is_night"] == "True") & 
             ((df["morning_rush"] == "True") | (df["evening_rush"] == "True"))).sum()
timecheck

np.int64(0)

In [11]:
df = clean.convert_column_types(
    df,
    bool_cols=["injured"],
    int_cols=["tu1_vehicle_size", "tu2_vehicle_size",
              "vehicle_size_diff", "abs_vehicle_size_diff"],
    float_cols=["lat", "lon"],
    category_cols=["tu1_vehicle_type_binned", "tu2_vehicle_type_binned", 
                   "light_condition", "road_condition"],
    string_cols=["incident_address", "top_traffic_accident_offense"]
)

df.dtypes

top_traffic_accident_offense            string
first_occurrence_date                  float64
incident_address                        string
lon                                    float64
lat                                    float64
road_description                           str
road_condition                        category
light_condition                       category
datetime                        datetime64[ns]
date                                    object
time                                       str
injured                                boolean
tu1_vehicle_type_binned               category
tu2_vehicle_type_binned               category
tu1_driver_action_binned                   str
tu2_driver_action_binned                   str
tu1_human_factor_binned                    str
tu2_human_factor_binned                    str
vehicle_type_match                         str
tu1_vehicle_size                         Int64
tu2_vehicle_size                         Int64
vehicle_size_

## Temporal features

In [12]:
df = features.create_time_variables(df)

## Presence

In [13]:
df = features.presence(df)

In [14]:
df["light_condition"] = (
    df["light_condition"]
    .str.replace(" ", "", regex=False)
)

## Highway

In [15]:
df = features.create_highway_indicator(df)

In [16]:
df["is_highway"].value_counts(normalize=True)

is_highway
False    0.807366
True     0.192634
Name: proportion, dtype: Float64

In [17]:
df.loc[
    df["is_highway"],
    ["incident_address", "is_highway"]
].sample(20, random_state=67)

,incident_address,is_highway
77865,W 6TH AVE / I25 HWYSB,True
242296,I25 HWYNB / W 6TH AVE,True
87936,N NORTHFIELD QUEBEC ST / I270 HWYEB,True
133257,SB E470 TO EB PENA RAMP / PENA BLVD,True
132694,I225 HWY_SB,True
8612,W COLFAX AVE / I25 HWYNB,True
245338,I225 HWYSB / I25 HWYSB,True
189194,I25 HWYSB / N SPEER BLVD,True
98346,E 56TH AVE / PENA BLVD INBOUND,True
68661,I25 HWYSB / I70 HWYEB,True


## speedlimit with coordinates

https://opendata-geospatialdenver.hub.arcgis.com/datasets/geospatialDenver::street-centerlines/about

Which road line is closest to each crash point?

In [18]:
roads_gdf = gpd.read_file("/Users/Local/REPOSITORIES/injury_risk_classifier/data/raw/ODC_TRANS_STREET_L_5394477846909849941.geojson")

In [19]:
roads_gdf.head()

,OBJECTID,FNODE,TNODE,STREETID,MASTERID,L_F_ADD,L_T_ADD,R_F_ADD,R_T_ADD,PREFIX,...,SNOWROUTE_ID,PRIORITY,RSPROUTE,PCI,created_user,created_date,last_edited_user,last_edited_date,GlobalID,geometry
0,3339107,11606.0,11724,830.0,10249,1600.0,1698.0,1601.0,1699.0,S,...,40101.0,Super A,NaN,NaN,STREETS,"Thu, 08 Jun 2017 00:00:00 GMT",220181,"Mon, 28 Apr 2025 15:21:34 GMT",00a43daa-5837-4f2b-b697-a2bc8109a83b,"LINESTRING (-104.94065 39.68749, -104.94069 39..."
1,3339108,19737.0,19728,3.0,30725,0.0,0.0,22878.0,22948.0,E,...,NaN,NaN,NaN,NaN,STREETS,"Thu, 17 Sep 2009 00:00:00 GMT",137161,"Wed, 17 Dec 2025 20:35:22 GMT",00bb22aa-fe8e-40d7-9ef3-0caaf41d74ed,"LINESTRING (-104.7206 39.8127, -104.7206 39.81..."
2,3339109,4221.0,4220,284.0,13503,4301.0,4399.0,4300.0,4398.0,E,...,NaN,NaN,RS206,NaN,STREETS,"Wed, 02 May 2018 00:00:00 GMT",220181,"Mon, 28 Apr 2025 15:21:34 GMT",011ccd08-892c-4f0b-a0f0-d5e12ea1bd93,"LINESTRING (-104.9372 39.76016, -104.93666 39...."
3,3339110,2910.0,2905,116.0,1276,3901.0,3999.0,3900.0,3998.0,E,...,30704.0,A,NaN,NaN,STREETS,"Wed, 06 Feb 2008 00:00:00 GMT",220181,"Mon, 28 Apr 2025 15:21:34 GMT",0124b7bc-0346-4071-80fe-ff0d77ac9c2f,"LINESTRING (-104.94169 39.77288, -104.94099 39..."
4,3339111,4206.0,3985,282.0,13817,3001.0,3199.0,3000.0,3198.0,N,...,NaN,NaN,RS206,NaN,STREETS,"Thu, 17 Jun 2004 00:00:00 GMT",220181,"Mon, 28 Apr 2025 15:21:34 GMT",017b36bd-337b-45f2-b3ec-e39f34a0b48b,"LINESTRING (-104.92111 39.76018, -104.92111 39..."


In [20]:
roads_gdf = roads_gdf[["SPEEDLIMIT", "FULLNAME", "geometry"]]

In [21]:
crashes_gdf = geo.make_crash_points(df)

In [22]:
crashes_with_roads = geo.convert_to_meters(crashes_gdf, roads_gdf)

In [23]:
crashes_with_roads = geo.match_quality(crashes_with_roads, "distance_to_road_m")

In [24]:
crashes_with_roads = crashes_with_roads.rename(columns={
    "SPEEDLIMIT": "speed_limit",
    "FULLNAME":"road_name"
})

In [25]:
crashes_with_roads = geo.add_speed_limit_features(crashes_with_roads, "speed_limit")

In [26]:
cols_to_check = [
    "road_name",
    "incident_address",
    "lat",
    "lon",
    "distance_to_road_m",
    "speed_limit"
]

crashes_with_roads[cols_to_check].sample(30, random_state=67)

,road_name,incident_address,lat,lon,distance_to_road_m,speed_limit
77858,E MONTVIEW BLVD,N MONACO ST / E MONTVIEW BLVD,39.747430,-104.912814,0.105824,30.0
110388,N CANOSA CT,2600 BLOCK W 8TH AVE,39.728977,-105.019489,32.085048,25.0
27435,N FEDERAL BLVD,EB 6TH TO FEDERAL RAMP / N FEDERAL BLVD,39.725140,-105.025029,0.092746,35.0
236503,W VIRGINIA AVE,W VIRGINIA AVE / S DECATUR ST,39.707514,-105.022707,0.036432,25.0
253009,CURTIS ST,19TH ST / CURTIS ST,39.750005,-104.991199,0.110110,25.0
19898,INTERSTATE 25,I25 HWYSB / N SPEER BLVD,39.755367,-105.012696,0.029676,55.0
179451,INTERSTATE 70,I70 HWYEB / N BRIGHTON BLVD,39.779882,-104.967168,0.115490,55.0
140008,PRIVATE RD,8260 E NORTHFIELD BLVD,39.782860,-104.892197,23.937633,25.0
258889,S DOWNING ST,I25 HWYSB / S DOWNING ST,39.689822,-104.973427,0.262382,30.0
228326,W ARKANSAS AVE,5125 W FLORIDA AVE,39.690620,-105.052088,82.099276,25.0


In [27]:
processed_data_sample = crashes_with_roads.sample(50, random_state=67)
processed_data_sample = processed_data_sample.drop(columns="geometry")

In [28]:
crashes_with_roads.shape

(271582, 50)

In [ ]:
# sample pushed to repo
# processed_data_sample.to_csv("processed_data_sample.csv", 
#                             index = False)

# saved on local machine 
#processed_data_sample.to_parquet("crashes_model_data.parquet",
#                                 index = False)